# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Object, not dict; access as attributes
print(f"Dataset: {getattr(metadata, 'name', '')}\nDescription: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. 

Here, we print all available RecordSets, their `@id`, and their Fields/Columns by `@id`. All references are shown by `@id` as required.

In [ ]:
# List available record sets and fields by @id

record_sets = list(dataset.record_sets)
print(f"Number of record sets found: {len(record_sets)}\n---\n")

# Show each RecordSet and its fields
for rs in record_sets:
    print(f"RecordSet @id: {getattr(rs, '@id', None) or getattr(rs, 'id', None)}")
    print(f"  Name: {getattr(rs, 'name', None)}")
    # Show declared fields (by @id)
    fields = getattr(rs, 'fields', [])
    if fields:
        print("  Fields (by @id):")
        for fld in fields:
            print(f"    - {getattr(fld, '@id', getattr(fld, 'id', None))}")
    columns = getattr(rs, 'columns', [])
    if columns:
        print("  Columns (by @id):")
        for col in columns:
            print(f"    - {getattr(col, '@id', getattr(col, 'id', None))}")
    print('---')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field/column `@id`s from the overview.

Below, we extract all record sets by `@id`. You can adjust the list to select a subset.

In [ ]:
# Collect all record set @ids
record_set_ids = [getattr(rs, '@id', getattr(rs, 'id', None)) for rs in record_sets]
print(f"RecordSet @ids: {record_set_ids}")

dataframes = {}
for rsid in record_set_ids:
    # Load all records for the current record set
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
        else:
            df = pd.DataFrame()
    except Exception as e:
        print(f"Error loading {rsid}: {e}")
        df = pd.DataFrame()
    dataframes[rsid] = df

# For demonstration, use the first record set that contains data:
main_rsid = None
for rsid, df in dataframes.items():
    if not df.empty:
        main_rsid = rsid
        break
if main_rsid:
    print(f"Fields/columns in {main_rsid}:")
    print(dataframes[main_rsid].columns.tolist())
    display(dataframes[main_rsid].head())
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes, to prepare it for further analysis.

### Choose a Numeric Field and Group Field by their `@id`
Replace `numeric_field_id` and `group_field_id` with actual column names (preferably the `@id` or mapped name) from the previous step.

In [ ]:
if main_rsid:
    df = dataframes[main_rsid]
    print(f"Data columns: {list(df.columns)}")
    # Try to guess a numeric field
    numeric_candidates = df.select_dtypes(include=["number"]).columns.tolist()
    print(f"Numeric candidate fields: {numeric_candidates}")

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
    else:
        print('No numeric fields found; skipping EDA.')
        numeric_field_id = None

    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        print(f"Filtering records with {numeric_field_id} > {threshold:.3f}")
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a group-able (categorical) field
        group_candidates = df.select_dtypes(include=["object", "category"]).columns.difference([numeric_field_id]).tolist()
        group_field_id = group_candidates[0] if group_candidates else None

        if group_field_id:
            print(f"Grouping data by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("No main DataFrame is available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below is a basic visualization using matplotlib if available. Adjust fields as appropriate.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rsid and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        order = df[group_field_id].value_counts().index[:10].tolist()
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(order)], order=order)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the dataset using the `mlcroissant` library and explored its structure and schema via `@id` references.
- We extracted key record sets and processed one for exploratory data analysis.
- We demonstrated field normalization and basic filtering by a numeric variable.
- Visualizations revealed basic data distributions and relationships.
- For deeper analysis, consult the Croissant schema or review additional fields/record sets relevant to your research question.